# DatensEE Quickstart — Colab / Notebook

Export Earth Engine imagery at scale via Cloud Dataflow, directly from a notebook.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/michaelfdewitt/datensee/blob/master/notebooks/datensee_quickstart.ipynb)

## 1. Install DatensEE

In [ ]:
!pip install -q "datensee[all] @ git+https://github.com/michaelfdewitt/datensee.git#subdirectory=cli"

## 2. Authenticate

In Colab, this triggers the interactive Google auth flow.
Outside Colab, ensure ADC is configured (`gcloud auth application-default login`).

In [ ]:
import datensee
from datensee import notebook

notebook.ensure_auth()

## 3. Define the export

Use the built-in demo (Landsat 9 NDVI over SF Bay Area) or define your own
expression with `ee.serializer.encode(image, for_cloud_api=True)`.

In [ ]:
from datensee.api import _DEMO_EXPRESSION, _DEMO_REGION

# For custom expressions:
# import ee
# ee.Initialize(project="your-project")
# image = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate('2023-06-01', '2023-09-01').median().normalizedDifference(['SR_B5', 'SR_B4'])
# expression = ee.serializer.encode(image, for_cloud_api=True)
# region = ee.Geometry.Rectangle([-122.5, 37.75, -122.25, 38.0]).getInfo()

expression = _DEMO_EXPRESSION
region = _DEMO_REGION

PROJECT = "your-gcp-project"  # <-- change this

## 4. Preview the tile grid

In [ ]:
grid = datensee.tile(region, scale=30.0)
print(f"{len(grid.tiles)} tiles at {grid.scale_meters} m/px")

notebook.display_tile_grid(grid, region)

## 5. Estimate costs

In [ ]:
from datensee.estimate import estimate_cost
from datensee.config import PipelineConfig, OutputConfig, RunnerConfig

# Build a quick config for estimation
from datensee.expression import clip_expression
config = PipelineConfig(
    ee_expression=clip_expression(expression, region),
    gee_project=PROJECT,
    tile_grid=grid,
    output=OutputConfig(output_path="gs://your-bucket/output"),
    runner=RunnerConfig(mode="local"),
)
est = estimate_cost(config)

notebook.display_estimate(est, config)

## 6. Run the export

For small regions, use `runner="local"` with a local output directory.
For large regions, use `runner="dataflow"` with a GCS output path.

In [ ]:
result = datensee.export(
    ee_expression=expression,
    region=region,
    project=PROJECT,
    output="./datensee-output",
    runner="local",
)

print(f"Tiles OK: {result.tiles_ok}")
print(f"Duration: {result.duration_seconds:.1f}s")
print(f"VRT: {result.vrt_path}")

## 7. Monitor a Dataflow job (optional)

If you used `runner="dataflow"`, poll the job with an HTML status display.

In [ ]:
# Uncomment to monitor a Dataflow job:
# final_state = notebook.display_job_progress(
#     result.job_id,
#     project=PROJECT,
#     region="us-central1",
# )

## 8. Preview output tiles

In [ ]:
# Requires datensee[eval] (rasterio)
# notebook.preview_tiles("./datensee-output", result.config, n=4)

## 9. Validate output (optional)

In [ ]:
# from datensee.eval import validate_output
# report = validate_output("./datensee-output", result.config)
# print(report.render())